In [ ]:
# Pre analysis all sublibrary and calculate TSS enrichment, doublets scores and etc 
# Get valid and high-quality barcodes

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "30"  # 设置为适合你CPU核心数的值

In [ ]:
import snapatac2 as snap
import numpy as np
import pandas as pd
import scanpy as sc
import polars as pl
import os 
snap.__version__

In [ ]:
os.chdir('QC_snapATAC2')

In [ ]:
myCustomGenome = snap.genome.Genome(fasta='/home/sunkeyong/softeware/ref_tree_shrew/TreeShrew_scATAC_ref/fasta/genome.fa', 
                                    annotation='/home/sunkeyong/softeware/ref_tree_shrew/TreeShrew_scATAC_ref/genes/genes.gtf')

In [ ]:
# 创建一个样本数据框以路径和样本名称初始化
sample= ["Sublibrary_Sample.list"]
path = ["pathway/Sublibrary_Sample.list.bed.gz"]

In [ ]:
%%time
data = snap.pp.import_data(
    path,
    file=[name + '.h5ad' for name in sample],
    chrom_sizes=myCustomGenome,
    sorted_by_barcode=True,
    min_num_fragments=5,
    n_jobs=8,
)
data

In [ ]:
snap.metrics.tsse(data, myCustomGenome, n_jobs=8)
snap.pp.add_tile_matrix(data, bin_size=500, n_jobs=8)
snap.pp.select_features(data, n_features=500000, n_jobs=8)
snap.pp.scrublet(data, n_jobs=3)
data

In [ ]:
# collect cell information

In [ ]:
sample= ["Sublibrary_Sample.h5ad"]

In [ ]:
data = snap.read(sample[10])

In [ ]:
data

In [ ]:
list_dfs = []
for x in range(len(sample)):
    data = snap.read(sample[x])
    df_temp = pd.DataFrame({'n_fragment':data.obs['n_fragment'],
                   'frac_dup':data.obs['frac_dup'],
                   'frac_mito':data.obs['frac_mito'],
                   'tsse':data.obs['tsse'],
                   'doublet_probability':data.obs['doublet_probability'],
                   'doublet_score':data.obs['doublet_score']},
                   index = data.obs_names)
    list_dfs.append(df_temp)
df_all = pd.concat(list_dfs)

In [ ]:
df_all.to_csv('/home/sunkeyong/Multispecies_analysis/M_TreeShrew/Process_code/snapATAC2_df_obs.csv')